# BharatStock AI V102 — Leakage-Controlled Model Validation (Google Colab)

**Purpose:** test whether candidate models add genuine out-of-sample value over simple baselines on the supplied NSE OHLCV data.

This notebook does **not** promise 60% accuracy and does not automatically replace the live API model. It produces evidence first. Run every cell from top to bottom and retain the generated CSV/JSON outputs for your project viva.

**Input:** a CSV with `date` or `timestamp`, `symbol`, `open`, `high`, `low`, `close`, `volume`. Upload the latest daily OHLCV export from BharatStock AI. Do not upload API tokens.

## 1. Install dependencies

In [ ]:
!pip -q install scikit-learn xgboost

## 2. Upload the daily OHLCV CSV

In [ ]:
from google.colab import files
uploaded = files.upload()
csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
if not csv_files:
    raise ValueError("Upload one OHLCV CSV file.")
CSV_PATH = csv_files[0]
print("Using:", CSV_PATH)

## 3. Load, normalize, and validate data

In [ ]:
import json, warnings, math
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, brier_score_loss
from sklearn.impute import SimpleImputer

raw = pd.read_csv(CSV_PATH)
raw.columns = [str(c).strip().lower() for c in raw.columns]
date_col = "date" if "date" in raw.columns else ("timestamp" if "timestamp" in raw.columns else None)
required = {"symbol","open","high","low","close","volume"}
if date_col is None or not required.issubset(raw.columns):
    raise ValueError(f"Required columns: date/timestamp, {sorted(required)}. Found: {raw.columns.tolist()}")
raw["date"] = pd.to_datetime(raw[date_col], errors="coerce", utc=True).dt.tz_convert(None).dt.normalize()
for c in ["open","high","low","close","volume"]:
    raw[c] = pd.to_numeric(raw[c], errors="coerce")
raw["symbol"] = raw["symbol"].astype(str).str.upper().str.strip()
raw = raw.dropna(subset=["date","symbol","open","high","low","close","volume"])
raw = raw[(raw[["open","high","low","close"]] > 0).all(axis=1) & (raw["volume"] >= 0)]
raw = raw.sort_values(["symbol","date"]).drop_duplicates(["symbol","date"], keep="last").reset_index(drop=True)
print("Rows:", len(raw), "| Symbols:", raw.symbol.nunique(), "| Dates:", raw.date.min(), "to", raw.date.max())
display(raw.groupby("symbol").agg(rows=("date","size"),first_date=("date","min"),last_date=("date","max")).reset_index())

## 4. Causal feature engineering

All indicators use data available at the end of session *t*. The target is whether close at *t+1* is above close at *t*. The last row per symbol has no target and is excluded from evaluation.

In [ ]:
def make_features(g):
    g = g.sort_values("date").copy()
    c,o,h,l,v = g.close, g.open, g.high, g.low, g.volume
    prev = c.shift(1)
    ret1 = c.pct_change()
    g["ret_1"] = ret1
    for n in [5,10,20]:
        g[f"ret_{n}"] = c.pct_change(n)
    g["vol_5"] = ret1.rolling(5).std()
    g["vol_20"] = ret1.rolling(20).std()
    g["sma20_gap"] = c / c.rolling(20).mean() - 1
    g["sma50_gap"] = c / c.rolling(50).mean() - 1
    g["ema20_gap"] = c / c.ewm(span=20, adjust=False).mean() - 1
    delta = c.diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = -delta.clip(upper=0).rolling(14).mean()
    rs = gain / loss.replace(0,np.nan)
    g["rsi14"] = (100 - 100/(1+rs)).fillna(50).clip(0,100)
    ema12 = c.ewm(span=12, adjust=False).mean()
    ema26 = c.ewm(span=26, adjust=False).mean()
    macd = ema12 - ema26
    g["macd_pct"] = macd / c
    g["macd_signal_pct"] = macd.ewm(span=9, adjust=False).mean() / c
    tr = pd.concat([(h-l),(h-prev).abs(),(l-prev).abs()],axis=1).max(axis=1)
    g["atr_pct"] = tr.rolling(14).mean() / c
    g["range_pct"] = (h-l) / c
    g["close_location"] = ((c-l)/(h-l).replace(0,np.nan)).clip(0,1)
    g["gap_pct"] = o / prev - 1
    g["volume_ratio20"] = v / v.rolling(20).median().replace(0,np.nan)
    g["target"] = (c.shift(-1) > c).astype(float)
    g.loc[g.index[-1], "target"] = np.nan
    return g.replace([np.inf,-np.inf],np.nan)

data = pd.concat([make_features(g) for _,g in raw.groupby("symbol")], ignore_index=True)
# Use NIFTY context only if an index series exists in the uploaded file.
index_candidates = [s for s in data.symbol.unique() if "NIFTY" in s or s in ["^NSEI","NIFTY 50","NIFTY50"]]
if index_candidates:
    idx = data[data.symbol == index_candidates[0]][["date","ret_1","ret_5","vol_20"]].rename(
        columns={"ret_1":"nifty_ret1","ret_5":"nifty_ret5","vol_20":"nifty_vol20"})
    data = data.merge(idx, on="date", how="left", validate="many_to_one")
else:
    data["nifty_ret1"] = np.nan
    data["nifty_ret5"] = np.nan
    data["nifty_vol20"] = np.nan

FEATURES = ["ret_1","ret_5","ret_10","ret_20","vol_5","vol_20","sma20_gap","sma50_gap",
            "ema20_gap","rsi14","macd_pct","macd_signal_pct","atr_pct","range_pct",
            "close_location","gap_pct","volume_ratio20"]
MARKET_FEATURES = FEATURES + ["nifty_ret1","nifty_ret5","nifty_vol20"]
data = data.sort_values(["date","symbol"]).reset_index(drop=True)
print("NIFTY context available:", bool(index_candidates), index_candidates[:1])
display(data[["date","symbol","close","target"]].tail())

## 5. Define regimes without looking into the future

Regime labels are derived from trailing 20-session volatility within each symbol. They are used for subgroup reporting, not as future information. The primary comparison remains chronological and pooled.

In [ ]:
data["regime"] = data.groupby("symbol")["vol_20"].transform(
    lambda s: pd.qcut(s.rank(method="first"), 3, labels=["low_vol","mid_vol","high_vol"])
)

## 6. Chronological walk-forward evaluation

The same global date folds are used for every model. Training rows whose next-session label crosses the test boundary are purged by excluding the final training date. Models are refit in each fold. No random train/test split is used.

In [ ]:
MODEL_FACTORIES = {
    "LogisticRegression": lambda: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight="balanced", C=0.1, random_state=42)),
    "RandomForest": lambda: make_pipeline(SimpleImputer(strategy="median"),
        RandomForestClassifier(n_estimators=250, max_depth=6, min_samples_leaf=8,
            max_features="sqrt", class_weight="balanced_subsample", random_state=42, n_jobs=-1)),
    "ExtraTrees": lambda: make_pipeline(SimpleImputer(strategy="median"),
        ExtraTreesClassifier(n_estimators=250, max_depth=6, min_samples_leaf=8,
            max_features="sqrt", class_weight="balanced", random_state=43, n_jobs=-1)),
    "HistGradientBoosting": lambda: make_pipeline(SimpleImputer(strategy="median"),
        HistGradientBoostingClassifier(max_iter=120, learning_rate=.04, max_leaf_nodes=7,
            l2_regularization=2.0, random_state=44))
}
try:
    from xgboost import XGBClassifier
    MODEL_FACTORIES["XGBoost"] = lambda: make_pipeline(SimpleImputer(strategy="median"),
        XGBClassifier(n_estimators=250, max_depth=3, learning_rate=.03, subsample=.8,
            colsample_bytree=.8, reg_lambda=5, eval_metric="logloss", random_state=45, n_jobs=2))
except Exception as e:
    print("XGBoost unavailable:", e)

def make_global_folds(frame, n_folds=5):
    dates = np.array(sorted(frame.date.dropna().unique()))
    initial = max(100, int(len(dates)*0.50))
    remaining = len(dates)-initial
    if remaining < n_folds*10:
        raise ValueError("Not enough unique dates for 5 chronological folds.")
    sizes = np.full(n_folds, remaining//n_folds, dtype=int)
    sizes[:remaining % n_folds] += 1
    folds=[]; cursor=initial
    for size in sizes:
        test_dates=dates[cursor:cursor+size]
        if len(test_dates):
            # Exclude one date immediately before test start: its target uses first test close.
            train_dates=dates[:max(0,cursor-1)]
            folds.append((set(train_dates),set(test_dates)))
        cursor += size
    return folds

def metrics(y, pred, prob):
    out={"accuracy":accuracy_score(y,pred),
         "balanced_accuracy":balanced_accuracy_score(y,pred),
         "f1":f1_score(y,pred,zero_division=0)}
    if len(np.unique(y)) == 2:
        out["auc"]=roc_auc_score(y,prob)
        out["brier"]=brier_score_loss(y,prob)
    else:
        out["auc"]=np.nan; out["brier"]=np.nan
    return out

def run_walk_forward(frame, feature_cols, label):
    d=frame.dropna(subset=["target"]).copy()
    d["target"]=d.target.astype(int)
    d=d.dropna(subset=["date"])
    folds=make_global_folds(d)
    rows=[]
    for fi,(train_dates,test_dates) in enumerate(folds,1):
        tr=d[d.date.isin(train_dates)].dropna(subset=feature_cols)
        te=d[d.date.isin(test_dates)].dropna(subset=feature_cols)
        if tr.empty or te.empty: continue
        ytr=tr.target; yte=te.target
        majority=int(ytr.value_counts().idxmax())
        rows.append({"fold":fi,"model":"MajorityBaseline","feature_set":label,
                     **metrics(yte,np.full(len(yte),majority),np.full(len(yte),majority,dtype=float))})
        for name,factory in MODEL_FACTORIES.items():
            model=factory()
            model.fit(tr[feature_cols],ytr)
            pred=model.predict(te[feature_cols])
            try:
                classes=list(model[-1].classes_) if hasattr(model,"steps") else list(model.classes_)
                proba=model.predict_proba(te[feature_cols])
                prob=proba[:,classes.index(1)] if 1 in classes else np.zeros(len(te))
            except Exception:
                prob=pred.astype(float)
            rows.append({"fold":fi,"model":name,"feature_set":label,**metrics(yte,pred,prob)})
    return pd.DataFrame(rows)

basic_results = run_walk_forward(data, FEATURES, "stock_only")
if data[["nifty_ret1","nifty_ret5","nifty_vol20"]].notna().any().any():
    market_results = run_walk_forward(data, MARKET_FEATURES, "stock_plus_nifty")
    results = pd.concat([basic_results,market_results],ignore_index=True)
else:
    results = basic_results.copy()
display(results.groupby(["feature_set","model"])[["accuracy","balanced_accuracy","auc","brier"]].mean().round(4).reset_index())

## 7. Regime-wise analysis (descriptive, not a new holdout)

This reports where the model's errors occur. It does not choose a model or threshold using the test folds.

In [ ]:
# Regime coverage and target balance
regime_summary = data.dropna(subset=["target","regime"]).groupby("regime",observed=True).agg(
    rows=("target","size"), up_rate=("target","mean"), avg_vol20=("vol_20","mean")).reset_index()
display(regime_summary)

## 8. Optional LSTM / GRU sequence experiment

Run only if you have enough data and Colab GPU is available. Sequences are created separately per symbol. The model is trained within each chronological fold, and the final sequence label is the next-session direction. This is an expensive benchmark; skip it if runtime is limited. These sequence results must not be confused with the live API model.

In [ ]:
RUN_NEURAL = False  # Change to True only after the classical benchmark completes.
if RUN_NEURAL:
    import tensorflow as tf
    from sklearn.preprocessing import StandardScaler
    SEQ_LEN=20
    def sequences_for_symbol(g, features):
        g=g.sort_values("date").dropna(subset=features+["target"])
        X=g[features].to_numpy(dtype="float32"); y=g.target.to_numpy(dtype="int32")
        dates=g.date.to_numpy()
        xs=[]; ys=[]; ds=[]
        for i in range(SEQ_LEN-1,len(g)):
            xs.append(X[i-SEQ_LEN+1:i+1]); ys.append(y[i]); ds.append(dates[i])
        return np.asarray(xs),np.asarray(ys),np.asarray(ds)
    # One representative feature set; report results as an optional research branch.
    seq_rows=[]
    for arch in ["LSTM","GRU"]:
        for fi,(train_dates,test_dates) in enumerate(make_global_folds(data),1):
            Xtr=[]; ytr=[]; Xte=[]; yte=[]
            for sym,g in data.groupby("symbol"):
                X,y,ds=sequences_for_symbol(g,FEATURES)
                trmask=np.isin(ds,list(train_dates)); temask=np.isin(ds,list(test_dates))
                if trmask.any(): Xtr.extend(X[trmask]); ytr.extend(y[trmask])
                if temask.any(): Xte.extend(X[temask]); yte.extend(y[temask])
            if len(Xtr)<100 or len(Xte)<20: continue
            Xtr=np.asarray(Xtr); Xte=np.asarray(Xte); ytr=np.asarray(ytr); yte=np.asarray(yte)
            # Fit scaling on training sequences only.
            scaler=StandardScaler().fit(Xtr.reshape(-1,Xtr.shape[-1]))
            Xtr=scaler.transform(Xtr.reshape(-1,Xtr.shape[-1])).reshape(Xtr.shape)
            Xte=scaler.transform(Xte.reshape(-1,Xte.shape[-1])).reshape(Xte.shape)
            inp=tf.keras.Input(shape=(SEQ_LEN,len(FEATURES)))
            layer=tf.keras.layers.LSTM(32)(inp) if arch=="LSTM" else tf.keras.layers.GRU(32)(inp)
            x=tf.keras.layers.Dropout(.25)(layer)
            out=tf.keras.layers.Dense(1,activation="sigmoid")(x)
            net=tf.keras.Model(inp,out)
            net.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss="binary_crossentropy")
            net.fit(Xtr,ytr,epochs=12,batch_size=64,verbose=0,
                    validation_split=0.1,callbacks=[tf.keras.callbacks.EarlyStopping(patience=3,restore_best_weights=True)])
            prob=net.predict(Xte,verbose=0).ravel(); pred=(prob>=.5).astype(int)
            seq_rows.append({"fold":fi,"model":arch,**metrics(yte,pred,prob)})
    neural_results=pd.DataFrame(seq_rows)
    display(neural_results.groupby("model")[["accuracy","balanced_accuracy","auc","brier"]].mean().round(4))
else:
    print("Neural benchmark skipped. Classical walk-forward results remain the primary evidence.")

## 9. Save reproducible evidence

In [ ]:
summary = results.groupby(["feature_set","model"])[["accuracy","balanced_accuracy","auc","brier","f1"]].agg(["mean","std"]).round(6)
summary.to_csv("V102_model_summary.csv")
results.to_csv("V102_fold_results.csv",index=False)
regime_summary.to_csv("V102_regime_summary.csv",index=False)
report = {
 "version":"V102",
 "target":"next-session close direction",
 "validation":"expanding chronological walk-forward, shared global date folds, one-date purge",
 "rows":int(len(data)),
 "symbols":sorted(data.symbol.unique().tolist()),
 "date_start":str(data.date.min().date()),
 "date_end":str(data.date.max().date()),
 "nifty_context_available":bool(index_candidates),
 "metrics_by_fold":results.replace({np.nan:None}).to_dict(orient="records"),
 "interpretation":"Choose no model unless it consistently beats the fold-specific majority baseline on balanced accuracy/AUC and remains useful after costs. Accuracy alone is insufficient.",
 "warning":"Historical research only; not a probability of profit, not investment advice, and not a live trading signal."
}
with open("V102_model_validation_report.json","w") as f: json.dump(report,f,indent=2,default=str)
files.download("V102_model_summary.csv")
files.download("V102_fold_results.csv")
files.download("V102_regime_summary.csv")
files.download("V102_model_validation_report.json")

## 10. Decision rule for the live model

Do **not** deploy the highest-scoring model just because it wins one fold. Promote a candidate only after:
1. It beats the fold-specific majority baseline on mean balanced accuracy and AUC across most folds.
2. Performance is not concentrated in one symbol or one market regime.
3. Probabilities are calibrated on validation data, not on the final holdout.
4. A cost-aware, non-overlapping paper-trading test is positive and robust to realistic fees/slippage.
5. A later untouched forward period confirms the result.

Until those conditions are met, the current API ensemble remains a research baseline. This notebook is a model-validation upgrade, not proof that the AI is finished.